⚠️ **Static Version Notice**

This is a static export of an interactive marimo notebook. Some features have been modified for compatibility:

- Interactive UI elements (sliders, dropdowns, text inputs) have been removed
- UI variable references have been replaced with default values
- Some cells may have been simplified or removed entirely

For the full interactive experience, please run the original marimo notebook (.py file) using:
```bash
uv run marimo edit notebook_name.py
```

---


In [3]:
# Ensure the large English spaCy model is available.
# It ships with 300-dimensional word vectors we need for embeddings/similarity.
import spacy

try:
    spacy.load("en_core_web_lg")
except OSError:
    from spacy.cli import download
    download("en_core_web_lg")


## Setup

First, we import the *spacy* library and load the large English model.

In [4]:
import spacy

nlp = spacy.load("en_core_web_lg")


Next, let's define a function to calculate word embeddings based on an input word:

In [5]:
def calculate_embedding(input_word):
    word = nlp(input_word)
    return word.vector



Let's try with the word 'apple'.  For brevity, only the first elements of the embedding vector are displayed:

In [6]:
calculate_embedding("apple")[:10]


array([-0.36391 ,  0.43771 , -0.20447 , -0.22889 , -0.14227 ,  0.27396 ,
       -0.011435, -0.18578 ,  0.37361 ,  0.75339 ], dtype=float32)

## Similarity
Let's add a function to calculate the similarity between two words based on their embeddings:

In [7]:
def calculate_similarity(word1, word2):
    return nlp(word1).similarity(nlp(word2))



Compare embeddings of words: 'apple' and 'car'

In [8]:
calculate_similarity("apple", "car")


0.2174709141254425

## Word Vector Arithmetic

One of the most striking properties of word embeddings is that **semantic relationships show up as directions in vector space**. The classic example is:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

Let's verify this. We'll build the composite vector on the left, then measure its cosine similarity to the vector for `"queen"`.


In [9]:
king = nlp("king").vector
man = nlp("man").vector
woman = nlp("woman").vector
queen = nlp("queen").vector

# king - man + woman  ~=  queen
composite = king - man + woman


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

sim_queen = cosine_similarity([composite], [queen])[0][0]
sim_king = cosine_similarity([composite], [king])[0][0]
sim_man = cosine_similarity([composite], [man])[0][0]
sim_woman = cosine_similarity([composite], [woman])[0][0]

print(f"cos(king - man + woman, queen) = {sim_queen:.3f}")


cos(king - man + woman, queen) = 0.788


----
## Sentence Embeddings

Finally, to calculate an embedding for a sentence, we can just average the embeddings of all the words in that sentence. We'll again use `spacy`, which does this averaging for us when we call `.similarity()` on a `Doc`.

Below, we compare a query against three candidate sentences — this is the core mechanic behind **semantic search** and **retrieval-augmented generation (RAG)**.


In [10]:
query = "What is the capital of France?"
info_1 = "The capital of France is Paris"
info_2 = "France is a beautiful country"
info_3 = "Today is very warm in New York City"
print("Response 1 Similarity: ", nlp(query).similarity(nlp(info_1)))
print("Response 2 Similarity: ", nlp(query).similarity(nlp(info_2)))
print("Response 3 Similarity: ", nlp(query).similarity(nlp(info_3)))


Response 1 Similarity:  0.9154103398323059
Response 2 Similarity:  0.8251080513000488
Response 3 Similarity:  0.797583818435669


Being able to quickly calculate similarities between a query and target information text is very powerful for Information Retrieval, especially when combined with Large Language Models trained for chat/question answering capabilities.